# Clase final: de descriptores clásicos a aprendizaje por transferencia

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NavaSergio/pdi-material-estudiantes/blob/main/docs/Unidad5/clase_transfer_learning/notebooks/clase_final_transfer_learning.ipynb)

**Pregunta guía:** ¿qué cambia cuando una imagen se representa con descriptores diseñados manualmente o con un modelo preentrenado?

Este notebook descarga Flower Photos, crea particiones estratificadas 70/15/15 y compara ambas representaciones con la misma regresión logística. En la primera ejecución necesita Internet para datos y pesos.

## 1. Preparación del entorno

En Colab se instalan dependencias faltantes. En local, instale `requirements.txt` desde la carpeta raíz de esta clase. Tras reiniciar un entorno de Colab, vuelva a ejecutar desde esta celda.

In [ ]:
import sys, subprocess, importlib.util
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'scikit-image>=0.22', 'seaborn>=0.13'])

import platform, random, tarfile, urllib.request, shutil, json
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
from skimage.color import rgb2gray
from skimage.feature import graycomatrix, graycoprops
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix, ConfusionMatrixDisplay, classification_report
from sklearn.decomposition import PCA
import torch
from torch import nn
from torch.utils.data import DataLoader, Dataset
from torchvision.models import resnet18, ResNet18_Weights

SEED = 2026
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    DEVICE = torch.device('cuda')
elif getattr(torch.backends, 'mps', None) and torch.backends.mps.is_available():
    DEVICE = torch.device('mps')
else:
    DEVICE = torch.device('cpu')
print(f'Entorno: {"Google Colab" if IN_COLAB else "local"} | Python {platform.python_version()} | dispositivo: {DEVICE}')
print('torch', torch.__version__)


## 2. Datos y particiones

`DEMO=True` limita el cálculo a una muestra estratificada para una clase de 60 minutos. Use `False` para el experimento completo. La lista de archivos y las tres particiones se guardan para que ambas rutas usen las mismas imágenes.

In [ ]:
# Ajuste reproducible
DEMO = True
MAX_PER_CLASS = 120       # se ignora cuando DEMO=False
ROOT = Path.cwd() if not IN_COLAB else Path('/content')
PROJECT = ROOT / 'clase_transfer_learning' if IN_COLAB else ROOT.parent
DATA_DIR = PROJECT / 'datos'
RESULTS = PROJECT / 'resultados'
FIGURES = RESULTS / 'graficos'
CACHE = RESULTS / 'caracteristicas'
for folder in (DATA_DIR, FIGURES, CACHE): folder.mkdir(parents=True, exist_ok=True)
archive = DATA_DIR / 'flower_photos.tgz'
flowers = DATA_DIR / 'flower_photos'
if not flowers.exists():
    if not archive.exists():
        urllib.request.urlretrieve('https://storage.googleapis.com/download.tensorflow.org/example_images/flower_photos.tgz', archive)
    with tarfile.open(archive, 'r:gz') as tar: tar.extractall(DATA_DIR)
classes = sorted(p.name for p in flowers.iterdir() if p.is_dir())
rows = [{'path': str(path), 'label': cls} for cls in classes for path in sorted((flowers/cls).glob('*.jpg'))]
df = pd.DataFrame(rows)
if DEMO:
    df = df.groupby('label', group_keys=False).sample(n=min(MAX_PER_CLASS, df.groupby('label').size().min()), random_state=SEED)
train_df, temp_df = train_test_split(df, test_size=.30, stratify=df.label, random_state=SEED)
val_df, test_df = train_test_split(temp_df, test_size=.50, stratify=temp_df.label, random_state=SEED)
for name, split in {'train':train_df, 'val':val_df, 'test':test_df}.items(): split.to_csv(RESULTS/f'{name}_split.csv', index=False)
display(pd.DataFrame({'total':df.label.value_counts().sort_index(), 'entrenamiento':train_df.label.value_counts(), 'validación':val_df.label.value_counts(), 'prueba':test_df.label.value_counts()}).fillna(0).astype(int))
ax = df.label.value_counts().sort_index().plot.bar(color='#0b5563', title='Imágenes por clase'); ax.set_ylabel('número de imágenes'); plt.show()


In [ ]:
# Mosaico real del conjunto descargado
fig, axes = plt.subplots(1, len(classes), figsize=(15, 3))
for ax, cls in zip(axes, classes):
    example = Path(df[df.label.eq(cls)].sample(1, random_state=SEED).path.iloc[0])
    ax.imshow(Image.open(example)); ax.set_title(cls); ax.axis('off')
fig.suptitle('Una imagen real por categoría'); plt.tight_layout(); fig.savefig(FIGURES/'mosaico_flores.png', dpi=160, bbox_inches='tight'); plt.show()


## 3. Ruta A: descriptores clásicos

Cada imagen aporta un histograma HSV, seis momentos de color, cuatro propiedades GLCM y siete momentos invariantes de Hu. Los momentos de Hu se calculan sobre una máscara simple; discuta cuándo esa segmentación resulta poco confiable.

In [ ]:
import cv2
def classic_features(path, size=(160,160)):
    rgb = np.array(Image.open(path).convert('RGB').resize(size))
    hsv = cv2.cvtColor(rgb, cv2.COLOR_RGB2HSV)
    hist = cv2.calcHist([hsv], [0,1,2], None, [8,8,4], [0,180,0,256,0,256]).ravel(); hist /= hist.sum() + 1e-12
    moments = np.r_[rgb.mean(axis=(0,1)), rgb.std(axis=(0,1))] / 255.0
    gray = (rgb2gray(rgb)*255).astype('uint8')
    glcm = graycomatrix(gray, distances=[1,3], angles=[0, np.pi/4], levels=256, symmetric=True, normed=True)
    texture = np.array([graycoprops(glcm, p).mean() for p in ('contrast','dissimilarity','homogeneity','energy')])
    _, mask = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    hu = -np.sign(cv2.HuMoments(cv2.moments(mask)).ravel()) * np.log10(np.abs(cv2.HuMoments(cv2.moments(mask)).ravel()) + 1e-12)
    return np.r_[hist, moments, texture, hu]

def extract_classic(split, name):
    cache = CACHE/f'classic_{name}_{"demo" if DEMO else "full"}.npz'
    if cache.exists(): return np.load(cache)['X']
    X = np.vstack([classic_features(p) for p in split.path])
    np.savez_compressed(cache, X=X); return X
Xc_train, Xc_val, Xc_test = [extract_classic(s, n) for s,n in [(train_df,'train'),(val_df,'val'),(test_df,'test')]]
print('Dimensión clásica:', Xc_train.shape[1])


## 4. Ruta B: características profundas con ResNet18

Se cargan pesos ImageNet, se reemplaza la capa clasificadora final por identidad y se congelan los parámetros. La red devuelve un vector de 512 componentes por imagen; solo la regresión logística aprende a partir de ellos.

In [ ]:
weights = ResNet18_Weights.DEFAULT
transform = weights.transforms()
model = resnet18(weights=weights)
model.fc = nn.Identity()
model.eval().to(DEVICE)
for parameter in model.parameters(): parameter.requires_grad = False

class PathDataset(Dataset):
    def __init__(self, paths): self.paths = list(paths)
    def __len__(self): return len(self.paths)
    def __getitem__(self, i): return transform(Image.open(self.paths[i]).convert('RGB'))

def extract_deep(split, name, batch_size=32):
    cache = CACHE/f'deep_{name}_{"demo" if DEMO else "full"}.npz'
    if cache.exists(): return np.load(cache)['X']
    loader = DataLoader(PathDataset(split.path), batch_size=batch_size, shuffle=False, num_workers=0)
    chunks = []
    with torch.inference_mode():
        for batch in loader: chunks.append(model(batch.to(DEVICE)).cpu().numpy())
    X = np.vstack(chunks); np.savez_compressed(cache, X=X); return X
Xd_train, Xd_val, Xd_test = [extract_deep(s, n) for s,n in [(train_df,'train'),(val_df,'val'),(test_df,'test')]]
print('Dimensión profunda:', Xd_train.shape[1])


## 5. Misma clasificación, misma evaluación

Seleccionamos `C` con validación y no consultamos el conjunto de prueba durante esa decisión. La prueba queda reservada para el informe final.

In [ ]:
y_train, y_val, y_test = train_df.label.to_numpy(), val_df.label.to_numpy(), test_df.label.to_numpy()
def fit_select(X_train, X_val, y_train, y_val):
    candidates = [0.1, 1, 10]
    scores, models = [], []
    for C in candidates:
        pipe = Pipeline([('scale', StandardScaler()), ('logreg', LogisticRegression(C=C, max_iter=2000, class_weight='balanced', random_state=SEED))])
        pipe.fit(X_train, y_train); pred = pipe.predict(X_val)
        scores.append(precision_recall_fscore_support(y_val, pred, average='macro', zero_division=0)[2]); models.append(pipe)
    best = int(np.argmax(scores)); print(f'C seleccionado: {candidates[best]} (F1 validación={scores[best]:.3f})')
    # Reajusta solo con entrenamiento + validación después de decidir C.
    final = Pipeline([('scale', StandardScaler()), ('logreg', LogisticRegression(C=candidates[best], max_iter=2000, class_weight='balanced', random_state=SEED))])
    final.fit(np.vstack([X_train,X_val]), np.r_[y_train,y_val]); return final
models = {'Clásica': fit_select(Xc_train,Xc_val,y_train,y_val), 'ResNet18': fit_select(Xd_train,Xd_val,y_train,y_val)}
Xtests = {'Clásica': Xc_test, 'ResNet18': Xd_test}
records, predictions = [], {}
for name, clf in models.items():
    pred = clf.predict(Xtests[name]); predictions[name] = pred
    p,r,f,_ = precision_recall_fscore_support(y_test, pred, average='macro', zero_division=0)
    records.append({'representación':name, 'exactitud':accuracy_score(y_test,pred), 'precisión macro':p, 'sensibilidad macro':r, 'F1 macro':f})
metrics = pd.DataFrame(records).set_index('representación'); display(metrics.round(3)); metrics.to_csv(RESULTS/'metricas_comparativas.csv')


In [ ]:
# Matrices de confusión reales
fig, axes = plt.subplots(1, 2, figsize=(13,5))
for ax, (name, pred) in zip(axes, predictions.items()):
    ConfusionMatrixDisplay(confusion_matrix(y_test, pred, labels=classes), display_labels=classes).plot(ax=ax, cmap='YlGnBu', colorbar=False, xticks_rotation=45)
    ax.set_title(name)
plt.tight_layout(); fig.savefig(FIGURES/'matrices_confusion.png', dpi=160, bbox_inches='tight'); plt.show()


## 6. PCA y análisis de errores

PCA ayuda a explorar las representaciones. La proyección bidimensional no sustituye las métricas del clasificador. Después, examinamos errores individuales para proponer causas observables.

In [ ]:
fig, axes = plt.subplots(1,2,figsize=(13,5))
for ax, (name, X) in zip(axes, {'Clásica':Xc_test, 'ResNet18':Xd_test}.items()):
    Z = PCA(n_components=2, random_state=SEED).fit_transform(StandardScaler().fit_transform(X))
    explained = PCA(n_components=2, random_state=SEED).fit(StandardScaler().fit_transform(X)).explained_variance_ratio_.sum()
    for cls in classes:
        mask = y_test == cls; ax.scatter(Z[mask,0], Z[mask,1], label=cls, alpha=.75, s=28)
    ax.set_title(f'{name} · varianza explicada: {explained:.1%}'); ax.legend(fontsize=8)
plt.tight_layout(); fig.savefig(FIGURES/'pca_representaciones.png', dpi=160, bbox_inches='tight'); plt.show()

name = 'ResNet18'; wrong = np.flatnonzero(predictions[name] != y_test)[:8]
fig, axes = plt.subplots(2,4,figsize=(13,6)); axes = axes.ravel()
for ax, idx in zip(axes, wrong):
    ax.imshow(Image.open(test_df.iloc[idx].path)); ax.set_title(f'real: {y_test[idx]}\npred.: {predictions[name][idx]}', fontsize=9); ax.axis('off')
for ax in axes[len(wrong):]: ax.axis('off')
plt.tight_layout(); fig.savefig(FIGURES/'errores_resnet18.png', dpi=160, bbox_inches='tight'); plt.show()


## 7. Conclusiones y preguntas

- ¿Qué propiedades visuales parece capturar mejor cada representación?
- ¿Qué error atribuirías al fondo, a la clase o a la representación?
- ¿Qué cambia si el dominio de las imágenes se aleja de ImageNet?
- ¿Cuándo justifica un fine-tuning su costo computacional y metodológico?

**Mensaje final:** ambas rutas convierten imágenes en vectores. La ingeniería clásica define explícitamente esas variables; el aprendizaje por transferencia reutiliza variables aprendidas en otra tarea.